# 15 — チューニング実験室

症状→仮説→1群変更→同一条件比較という順序を、簡単なMPCで練習します。

**前提**: `14_logging_and_diagnosis.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: MPCチューニング実験を上流PyMPCと同じリポジトリ配置・環境変数で行います。
# 目的: ワークスペースとPyMPCの場所を確定し、後続セルの実行条件を再現可能にします。
# OSに依存しないパス演算を行うためPathを読み込む。
from pathlib import Path
# 環境変数の設定にos、モジュール検索パスの設定にsysを使う。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化する。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけリポジトリルートへ移る。
if ROOT.name == "notebook_pympc":
    # 外部実装をROOT基準で参照できるよう親ディレクトリを採用する。
    ROOT = ROOT.parent
# 上流Quadruped-PyMPCの配置先をROOTから組み立てる。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動場所のまま実験を進めないよう実装の存在を検証する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同名モジュールの取り違えを防ぐため検索パス未登録時だけ処理する。
if str(PYMPC_ROOT) not in sys.path:
    # 現行リポジトリの実装を最優先でimportするため先頭へ追加する。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物の探索基準を未設定時だけ上流同梱ディレクトリへ合わせる。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画できるよう未設定時はEGLを選ぶ。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実験が参照するワークスペースを目視確認できるよう表示する。
print("workspace :", ROOT)
# 上流実装の参照先を目視確認できるよう表示する。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 推奨する調整順

1. model/単位/frame/脚順の誤りを除外
2. 実摩擦以下のMPC摩擦、GRF・torque上限を確認
3. gaitの周波数・duty・swing heightで運動学的余裕を作る
4. footholdと速度指令を確認
5. Q/Rを状態群ごとに変更
6. horizon、dt、solver option
7. 高度機能（積分器、foothold最適化、RTI等）

重みから始めると、構造上不可能な歩容を大きな力で追わせる危険があります。

In [2]:
# 背景: Q/R調整は追従誤差、入力energy、飽和率の交換条件として同一条件で比較します。
# 目的: 1次元MPCのQ×R格子を走査し、RMSE・入力energy・飽和率を定量化します。
# 予測軌道、二乗和、平方根の数値計算にNumPyを使う。
import numpy as np
# 入力制約付きMPC問題を解くためSciPyのminimizeを使う。
from scipy.optimize import minimize
# 試行結果を列名付き表として比較するためpandasを使う。
import pandas as pd

# 状態重みq、入力重みr、horizon長Nの1条件を20周期評価する。
def trial(q, r, N=10):
    # 初期状態-1、目標1、飽和回数0を同時に初期化する。
    x, target, sat = -1.0, 1.0, 0
    # 二乗追従誤差と入力二乗和を0から累積する。
    sqerr, effort = 0., 0.
    # 閉ループを20制御周期だけ反復して全条件の試験長をそろえる。
    for _ in range(20):
        # 現在状態xからの入力列uに対する有限時間costを定義する。
        def cost(u):
            # x[k+1]=x[k]+u[k]を累積し、shape (N+1,) の予測状態列を作る。
            xs = x + np.concatenate([[0.], np.cumsum(u)])
            # J=Σq(x-target)²+Σru²をスカラーで返す。
            return q*np.sum((xs-target)**2) + r*np.sum(u**2)
        # 各入力を[-0.3,0.3]に制約し、ゼロ列を初期値としてN段解を得る。
        sol = minimize(cost, np.zeros(N), bounds=[(-.3,.3)]*N).x
        # Receding horizon則に従い最適入力列の先頭だけを適用する。
        u = sol[0]
        # 数値許容を見込み|u|>0.299なら±0.3制約への飽和として数える。
        sat += abs(u) > .299
        # 離散力学x←x+uでPlant状態を1周期進める。
        x += u
        # 更新後の目標誤差二乗をRMSE計算用に累積する。
        sqerr += (x-target)**2
        # 適用入力u²を入力energy指標として累積する。
        effort += u*u
    # 20周期RMSE、入力energy、飽和周期率を同じ順序で返す。
    return np.sqrt(sqerr/20), effort, sat/20

# Q/R各条件の指標を蓄える空の行リストを作る。
rows = []
# 状態誤差を重くする度合いを3桁で走査する。
for q in [1, 10, 100]:
    # 入力使用を重くする度合いも3桁で走査する。
    for r in [0.1, 1, 10]:
        # 現在のQ/R組で閉ループ指標を計算する。
        rmse, effort, sat = trial(q, r)
        # 条件と3指標を対応付けた1行を結果へ追加する。
        rows.append({"Q":q, "R":r, "RMSE":rmse, "effort":effort, "sat_rate":sat})
# 全9条件をDataFrameに変換し、小数3桁でNotebookへ表示する。
pd.DataFrame(rows).round(3)

,Q,R,RMSE,effort,sat_rate
0,1,0.1,0.591,0.574,0.30
1,1,1.0,0.591,0.558,0.30
2,1,10.0,0.610,0.458,0.15
3,10,0.1,0.591,0.579,0.30
4,10,1.0,0.591,0.574,0.30
5,10,10.0,0.591,0.558,0.30
6,100,0.1,0.591,0.580,0.30
7,100,1.0,0.591,0.579,0.30
8,100,10.0,0.591,0.574,0.30


Qを増やしてRMSEが下がっても、飽和率と入力energyが悪化することがあります。
目的関数値だけでなくPlant側の制約指標を併記してください。

実験記録には「変更理由」「期待する向き」「副作用」「採否」を残します。
1 trialに複数群を変えると、改善の原因を学べません。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。